# dbt-for-apache-doris: 5 个 data-eng-bench Demo

这个 Notebook 依次运行 5 个可复现的 Doris + dbt Demo：每日订单、客户地域、广告合并、迟到订单增量和客户 Snapshot。每个 Demo 都会重建自己的专用 Database，执行 dbt model 和 Data Test，并运行 verifier。

运行前需要：

- 一个可访问的 Apache Doris 集群；
- 已安装 `dbt-for-apache-doris` 的 Python 环境；
- `mysql` 命令行客户端；
- Jupyter kernel 运行在包含 Doris 仓库的服务器上；Mac 可以通过 SSH 隧道在浏览器中访问。

默认连接 `127.0.0.1:9030`，用户为 `root`、密码为空。可以在启动 Jupyter 前设置 `DORIS_HOST`、`DORIS_PORT`、`DORIS_USER`、`DORIS_PASSWORD`、`DBT_BIN` 和 `MYSQL_BIN`。远程启动和 Mac SSH 隧道命令见同目录 `README.md`。

## 1. 初始化执行环境

下面的辅助函数只负责定位仓库、传递连接参数、运行已有 `run.sh` 和显示 Doris 查询结果。Demo 的建模逻辑仍保存在各自的 dbt project 中。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "extension/dbt-doris/examples").is_dir():
            return candidate
    raise FileNotFoundError(
        "请从 Apache Doris 仓库目录或其子目录启动 Jupyter。"
    )


repo_root = find_repo_root(Path.cwd().resolve())
examples_root = repo_root / "extension/dbt-doris/examples"
dbt_bin = os.environ.get("DBT_BIN") or shutil.which("dbt")
mysql_bin = os.environ.get("MYSQL_BIN") or shutil.which("mysql")
assert dbt_bin, "找不到 dbt，请设置 DBT_BIN。"
assert mysql_bin, "找不到 mysql，请设置 MYSQL_BIN。"

demo_env = os.environ.copy()
demo_env.update({
    "DBT_BIN": dbt_bin,
    "MYSQL_BIN": mysql_bin,
    "DORIS_HOST": os.environ.get("DORIS_HOST", "127.0.0.1"),
    "DORIS_PORT": os.environ.get("DORIS_PORT", "9030"),
    "DORIS_USER": os.environ.get("DORIS_USER", "root"),
    "DORIS_PASSWORD": os.environ.get("DORIS_PASSWORD", ""),
})


def run_command(command, cwd=None):
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=demo_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    assert return_code == 0, f"命令执行失败，退出码: {return_code}"


def run_demo(relative_path):
    project_dir = examples_root / relative_path
    assert (project_dir / "scripts/run.sh").is_file(), project_dir
    print(f"运行 Demo: {project_dir.name}")
    run_command([str(project_dir / "scripts/run.sh")], cwd=project_dir)


def query(sql):
    command = [
        mysql_bin,
        "-h", demo_env["DORIS_HOST"],
        "-P", demo_env["DORIS_PORT"],
        "-u", demo_env["DORIS_USER"],
        f"--password={demo_env['DORIS_PASSWORD']}",
        "-t",
        "-e", sql,
    ]
    run_command(command)


print(f"Repository: {repo_root}")
print(f"Doris: {demo_env['DORIS_HOST']}:{demo_env['DORIS_PORT']}")
print(f"dbt: {dbt_bin}")

In [ ]:
run_command([dbt_bin, "--version"])
query("select version() as doris_version; show backends;")

## 2. Demo 1：每日订单汇总

`dbt_demo_daily_source.orders` 经过状态过滤和按日聚合，生成 Doris Table `dbt_demo_daily.daily_order_summary`，再生成月度异步物化视图。这个 Demo 覆盖 Source、Table、Data Test、Duplicate Key、Range Partition、Hash Distribution 和 Async Materialized View。

In [ ]:
run_demo("data-eng-bench-daily-order-summary")
query("""
select order_date, order_count, total_revenue
from dbt_demo_daily.daily_order_summary
order by order_date;
select order_month, order_count, total_revenue
from dbt_demo_daily.monthly_order_summary_mv;
""")

## 3. Demo 2：客户地域分析

两个 Doris Database 中的地址和订单 Source 分别生成 staging View，再通过 `ref()` 合并成 `dbt_demo_geographic.fct_state_customers` Table。这个 Demo 覆盖跨 Database Source、View、Table、DAG 和重复构建。

In [ ]:
run_demo("data-eng-bench-doris-demos/geographic")
query("""
select state_province, customer_count, order_count, total_revenue, avg_order_value
from dbt_demo_geographic.fct_state_customers
order by state_province;
""")

## 4. Demo 3：广告数据标准化和合并

三个 CSV 先通过 Seed 写入 Doris，再生成三个去重 staging View，最后 `UNION ALL` 为统一 Table。这个 Demo 覆盖 Seed、`dbt deps`、`dbt_utils`、`QUALIFY`、View、Table 和 Data Test。

In [ ]:
run_demo("data-eng-bench-doris-demos/consolidate")
query("""
select source, ad_date, clicks, impressions, views, conversions
from dbt_demo_consolidate.int__ads_unified
order by source, ad_date;
""")

## 5. Demo 4：迟到订单 Incremental

脚本先进行首次全量 build，再插入订单 101 的新版本和新订单 104，执行 Incremental `merge`，最后做一次无输入变化 build。这个 Demo 覆盖 `is_incremental()`、`unique_key`、Unique Key Table、迟到数据和幂等运行。

In [ ]:
run_demo("data-eng-bench-doris-demos/incremental")
query("""
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id;
select order_date, order_count, total_revenue
from dbt_demo_incremental.daily_sales_summary
order by order_date;
""")

## 6. Demo 5：客户 Snapshot

第一次 Snapshot 写入两个当前客户。脚本随后修改客户 1、删除客户 2，再执行第二次 Snapshot 和当前维表。这个 Demo 覆盖 Check Strategy、SCD Type 2、`invalidate_hard_deletes` 和 Snapshot 下游 `ref()`。

In [ ]:
run_demo("data-eng-bench-doris-demos/snapshot")
query("""
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from;
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id;
""")

## 完成

前面的 5 个运行单元格全部成功，表示这些 Demo 的 dbt node、Doris 对象和 verifier 均已通过。每个 Demo 使用独立的专用 Database，可以单独重新执行。完整 model 和验收说明见 `extension/dbt-doris/docs/data-eng-bench-dbt-doris-demos.zh-CN.md`。